# 02 Parse Bronze Tables

Read downloaded ZIPs from the raw manifest, parse MMSDM CSV content, append Bronze Delta tables, write file audit rows, and quarantine malformed files.


## Configure Bronze Parse Run

This cell detects local versus Fabric runtime and defines parse parameters.


In [1]:
# Cell purpose: Configure Bronze Parse Run.
from datetime import datetime, timezone
from pathlib import Path
import csv
import importlib
import os
import sys
import uuid

try:
    spark
    is_local_run = False
except NameError:
    is_local_run = True

run_id = str(uuid.uuid4())
max_files_per_run = 500
fabric_quarantine_path = "Files/nemweb/quarantine"
local_output_folder = "data"
local_quarantine_path = "files/nemweb/quarantine"

print(f"run_id={run_id}")
print(f"runtime={'local' if is_local_run else 'fabric'}")


run_id=5e00cf00-1c91-4300-ae07-5a526dd644ca
runtime=local


## Resolve Runtime Paths

This cell resolves local or Fabric package paths before importing project modules.


In [2]:
# Cell purpose: Resolve Runtime Paths.
def find_repo_root(start: Path) -> Path:
    """Find the local repo root from a notebook working directory."""

    for candidate in (start, *start.parents):
        if (candidate / "src" / "nem_fabric").exists():
            return candidate
    return start


repo_root = None
local_output_root = None
package_paths = []

if is_local_run:
    repo_root = find_repo_root(Path.cwd())
    local_output_root = repo_root / local_output_folder
    package_paths.append(repo_root / "src")
else:
    package_paths.append(
        Path(os.getenv("FABRIC_NOTEBOOK_LIB_PATH", "/lakehouse/default/Files/libs"))
    )

for package_path in package_paths:
    if package_path.exists() and str(package_path) not in sys.path:
        sys.path.insert(0, str(package_path))

print(
    "Python search paths added:", [str(path) for path in package_paths if path.exists()]
)
if is_local_run:
    print(f"Local output root: {local_output_root}")


Python search paths added: ['c:\\Users\\brcol\\My Drive\\Documents\\!!!Resume\\Sample Work\\Fabric - NEMWeb Energy Market Data Pipeline (Lakehouse, PySpark, Delta, Power BI, Direct Lake)\\src']
Local output root: c:\Users\brcol\My Drive\Documents\!!!Resume\Sample Work\Fabric - NEMWeb Energy Market Data Pipeline (Lakehouse, PySpark, Delta, Power BI, Direct Lake)\data


## Import Parser and Define Helpers

This cell imports project modules after runtime paths are resolved, then defines local and Fabric helper functions.


In [3]:
# Cell purpose: Import Parser and Define Helpers.
import pandas as pd

from nem_fabric.common_mmsdm_parser import parse_zip_bytes

local_ingestion = (
    importlib.import_module("nem_fabric.local_ingestion") if is_local_run else None
)
windows_long_path = (
    local_ingestion._windows_long_path
    if is_local_run and local_ingestion is not None
    else lambda path: path
)

if is_local_run:
    if local_output_root is None:
        raise RuntimeError("local_output_root was not resolved for local run.")
else:
    pass


def table_exists(table_name: str) -> bool:
    """Return True when a Lakehouse table exists in the current Spark catalogue."""

    return spark.catalog.tableExists(table_name)


def read_zip_bytes(relative_zip_path: str) -> bytes:
    """Read ZIP bytes from the selected runtime storage root."""

    root = local_output_root if is_local_run else Path("/lakehouse/default")
    if root is None:
        raise RuntimeError("Local output root was not resolved.")
    zip_file_path = windows_long_path(root / relative_zip_path)
    with zip_file_path.open("rb") as file:
        return file.read()


def append_local_control_csv_rows(csv_path: Path, rows: list[dict]) -> None:
    """Append local control CSV rows using the project helper."""

    if local_ingestion is None:
        raise RuntimeError("local_ingestion was not imported for local run.")
    local_ingestion.append_csv_rows(csv_path, rows)


def write_local_parquet_part(parquet_part_path: Path, df: pd.DataFrame) -> None:
    """Write one deterministic local Parquet part file."""

    if local_ingestion is None:
        raise RuntimeError("local_ingestion was not imported for local run.")
    local_ingestion.write_parquet_part(parquet_part_path, df)


def local_parquet_part_name(*parts: str) -> str:
    """Build a deterministic local Parquet part filename."""

    if local_ingestion is None:
        raise RuntimeError("local_ingestion was not imported for local run.")
    return local_ingestion.safe_parquet_part_name(*parts)


if not is_local_run and not table_exists("raw_zip_manifest"):
    raise RuntimeError("raw_zip_manifest does not exist. Run notebook 01 first.")


## Select Unparsed ZIP Files

This cell reads the raw ZIP manifest and removes files already present in the file audit table, making Bronze parsing idempotent.


In [4]:
# Cell purpose: Select Unparsed ZIP Files.
if is_local_run:
    if local_output_root is None:
        raise RuntimeError("local_output_root was not resolved for local run.")
    manifest_csv_path = local_output_root / "tables" / "raw_zip_manifest.csv"
    audit_csv_path = local_output_root / "tables" / "raw_file_audit.csv"
    if not manifest_csv_path.exists():
        raise RuntimeError("Local manifest does not exist. Run notebook 01 first.")
    with manifest_csv_path.open("r", encoding="utf-8", newline="") as file:
        manifest_rows = [
            row for row in csv.DictReader(file) if row.get("status") == "downloaded"
        ]
    parsed_urls = set()
    if audit_csv_path.exists():
        with audit_csv_path.open("r", encoding="utf-8", newline="") as file:
            parsed_urls = {
                row["source_url"]
                for row in csv.DictReader(file)
                if row.get("source_url") and row.get("status") == "parsed"
            }
    zip_files_to_parse = [
        row for row in manifest_rows if row["source_url"] not in parsed_urls
    ]
    zip_files_to_parse = sorted(
        zip_files_to_parse, key=lambda row: row.get("file_datetime", "")
    )[:max_files_per_run]
else:
    manifest = spark.table("raw_zip_manifest").filter("status = 'downloaded'")
    if table_exists("raw_file_audit"):
        parsed = spark.table("raw_file_audit").select("source_url").distinct()
        manifest = manifest.join(parsed, on="source_url", how="left_anti")
    zip_files_to_parse = (
        manifest.orderBy("file_datetime").limit(max_files_per_run).collect()
    )

print(f"ZIP files selected for Bronze parsing: {len(zip_files_to_parse)}")


ZIP files selected for Bronze parsing: 193


## Parse ZIP Files and Build Audit Records

This cell parses each selected ZIP, preserves row-level metadata, collects Bronze DataFrames, and records parsing outcomes for auditability.


In [5]:
# Cell purpose: Parse ZIP Files and Build Audit Records.
audit_rows = []

def write_bronze_table(source_label: str, source_zip_name: str, df: pd.DataFrame) -> None:
    """Append one parsed DataFrame to its source-labelled Bronze table."""

    if df.empty:
        return
    if is_local_run:
        if local_output_root is None:
            raise RuntimeError("local_output_root was not resolved for local run.")
        parquet_part_name = local_parquet_part_name(
            Path(source_zip_name).stem,
            str(df["package_name"].iloc[0]) if "package_name" in df.columns else "unknown_package",
            str(df["table_name"].iloc[0]) if "table_name" in df.columns else "unknown_table",
        )
        write_local_parquet_part(
            local_output_root / "tables" / f"bronze_{source_label}" / parquet_part_name,
            df,
        )
    else:
        # MMSDM ZIPs contain multiple logical tables with different columns.
        # Merge schemas so later DISPATCH PRICE/REGIONSUM columns remain queryable.
        (
            spark.createDataFrame(df.astype(str))
            .write.format("delta")
            .mode("append")
            .option("mergeSchema", "true")
            .saveAsTable(f"bronze_{source_label}")
        )


for zip_file in zip_files_to_parse:
    parsed_at = datetime.now(timezone.utc).isoformat()
    status = "parsed"
    error_message = ""
    table_count = 0
    row_count = 0
    source_url = zip_file["source_url"] if is_local_run else zip_file.source_url
    source_name = zip_file["source_name"] if is_local_run else zip_file.source_name
    source_label = (
        zip_file.get("source_label", source_name)
        if is_local_run
        else getattr(zip_file, "source_label", source_name)
    )
    source_zip_name = zip_file["source_zip_name"] if is_local_run else zip_file.source_zip_name
    lakehouse_zip_path = zip_file["lakehouse_path"] if is_local_run else zip_file.lakehouse_path
    try:
        zip_bytes = read_zip_bytes(lakehouse_zip_path)
        tables = parse_zip_bytes(zip_bytes, source_url)
        table_count = len(tables)
        for table in tables:
            pdf = table.dataframe.copy()
            pdf["run_id"] = run_id
            pdf["source_name"] = source_name
            pdf["source_label"] = source_label
            pdf["bronze_loaded_datetime"] = parsed_at
            row_count += len(pdf)
            write_bronze_table(source_label, source_zip_name, pdf)
    except Exception as exc:
        status = "failed"
        error_message = str(exc)[:4000]
        try:
            root = local_output_root if is_local_run else Path("/lakehouse/default")
            quarantine_path = (
                local_quarantine_path if is_local_run else fabric_quarantine_path
            )
            if root is None:
                raise RuntimeError("Local output root was not resolved.")
            source_zip_file_path = windows_long_path(root / lakehouse_zip_path)
            quarantine_zip_file_path = windows_long_path(root / quarantine_path / source_zip_name)
            quarantine_zip_file_path.parent.mkdir(parents=True, exist_ok=True)
            quarantine_zip_file_path.write_bytes(source_zip_file_path.read_bytes())
        except Exception as quarantine_exc:
            error_message = f"{error_message}; quarantine_failed={quarantine_exc}"[
                :4000
            ]

    audit_rows.append(
        {
            "run_id": run_id,
            "source_name": source_name,
            "source_label": source_label,
            "source_url": source_url,
            "source_zip_name": source_zip_name,
            "lakehouse_path": lakehouse_zip_path,
            "parsed_datetime": parsed_at,
            "status": status,
            "table_count": table_count,
            "row_count_bronze": row_count,
            "error_message": error_message,
        }
    )


## Write Bronze and Audit Tables

This cell appends parsed MMSDM rows to Bronze Delta tables, creates source-specific Bronze subsets, and writes file audit results.


In [6]:
# Cell purpose: Write Audit Table.
if audit_rows:
    if is_local_run:
        if local_output_root is None:
            raise RuntimeError("local_output_root was not resolved for local run.")
        append_local_control_csv_rows(
            local_output_root / "tables" / "raw_file_audit.csv", audit_rows
        )
        print(f"Audit CSV: {local_output_root / 'tables' / 'raw_file_audit.csv'}")
        print(f"Audit rows written: {len(audit_rows)}")
    else:
        spark.createDataFrame(audit_rows).write.format("delta").mode(
            "append"
        ).saveAsTable("raw_file_audit")
        display(spark.createDataFrame(audit_rows))
else:
    print("No files required parsing.")


Audit CSV: c:\Users\brcol\My Drive\Documents\!!!Resume\Sample Work\Fabric - NEMWeb Energy Market Data Pipeline (Lakehouse, PySpark, Delta, Power BI, Direct Lake)\data\tables\raw_file_audit.csv
Audit rows written: 193
